# Tutorial 6 — Diagnosing Training Pathologies

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part II — Debugging & Observability**  
**Follows:** Tutorial 5 (Real-Time Training Dashboards)  
**Precedes:** Tutorial 7 (Data Pipelines for Pretraining)

---

## What This Tutorial Covers

This tutorial is a field guide. It does not introduce new abstractions — it
uses everything built in Tutorials 4 and 5 as diagnostic instruments and shows
what each failure mode looks like on the dashboard, why it happens
mathematically, and how to fix it.

The pathologies covered, ordered by frequency:

1. **Loss plateau** — the model stops improving and sits at a fixed loss
2. **Loss spike** — sudden sharp increase in loss, may or may not recover
3. **Loss divergence** — loss grows monotonically and does not recover
4. **Vanishing gradients** — early layers stop learning silently
5. **Exploding gradients** — gradient norms blow up, destabilizes training
6. **NaN propagation** — loss becomes NaN and the model is dead
7. **Dead neurons** — ReLU/GeLU units permanently silenced
8. **Overfitting** — train loss falls, eval loss rises

For each pathology:
- What the dashboard shows (loss curve, gradient norm, ratio chart)
- The mathematical reason it happens
- A minimal reproducible example that triggers it
- The fix

At the end: a `HealthReport` class that runs after every epoch, detects all
eight pathologies automatically, and prints a prioritized list of what to fix.

---

## Setup

All experiments use the nano GPT from Tutorial 2 and the training loop template
from Tutorial 5. We deliberately trigger each pathology — the code is designed
to break, so you can see what breaking looks like.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

from tutorial_02 import GPT, NanoGPTConfig
from tutorial_03 import Tokenizer
from tutorial_04 import GradientMonitor, LightweightMonitor
from training_logger import TrainingLogger

# Shared data setup
config = NanoGPTConfig()
tok    = Tokenizer.load('nano_tokenizer.json')
text   = open('tinyshakespeare.txt').read()
data   = torch.tensor(tok.encode(text), dtype=torch.long)
block  = config.max_seq_len
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_batch(split='train', batch_size=8):
    n  = int(0.9 * len(data))
    d  = data[:n] if split == 'train' else data[n:]
    ix = torch.randint(len(d) - block, (batch_size,))
    x  = torch.stack([d[i   : i+block  ] for i in ix]).to(device)
    y  = torch.stack([d[i+1 : i+block+1] for i in ix]).to(device)
    return x, y

@torch.no_grad()
def evaluate(model, n_batches=20):
    model.eval()
    losses = [model(*get_batch('val'))[1].item() for _ in range(n_batches)]
    model.eval()
    return float(np.mean(losses))

def run_experiment(
    model,
    optimizer,
    steps=300,
    label='experiment',
    clip=1.0,
):
    """
    Minimal training loop that records everything.
    Returns (train_losses, eval_losses, grad_norms, ratios).
    """
    monitor = LightweightMonitor(model, log_every=1)
    train_losses, eval_losses, grad_norms, ratios = [], [], [], []

    model.train()
    for step in range(steps):
        x, y  = get_batch()
        _, loss = model(x, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  [{label}] NaN/Inf loss at step {step} — stopping")
            train_losses.extend([float('nan')] * (steps - step))
            break

        optimizer.zero_grad()
        loss.backward()

        # Compute stats before clipping
        total_sq, layer_ratios = 0.0, []
        for name, mod in model.named_modules():
            if isinstance(mod, nn.Linear) and mod.weight.grad is not None:
                g = mod.weight.grad.norm().item()
                w = mod.weight.norm().item()
                total_sq += g ** 2
                layer_ratios.append(g / (w + 1e-8))
        gnorm = total_sq ** 0.5

        if clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        train_losses.append(loss.item())
        grad_norms.append(gnorm)
        ratios.append(float(np.mean(layer_ratios)) if layer_ratios else 0.0)

        if step % 100 == 0:
            el = evaluate(model)
            eval_losses.append((step, el))

    return train_losses, eval_losses, grad_norms, ratios


def plot_pathology(results: dict, title: str, filename: str):
    """
    Plot train loss + grad norm for multiple experiments side by side.
    `results` is {label: (train_losses, eval_losses, grad_norms, ratios)}.
    """
    fig, axes = plt.subplots(3, len(results), figsize=(5 * len(results), 9),
                             squeeze=False)
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for col, (label, (tl, el, gn, ra)) in enumerate(results.items()):
        steps = list(range(len(tl)))

        # Row 0: train loss
        axes[0][col].semilogy(steps, tl, color='#2196F3', lw=1.5, alpha=0.7)
        if el:
            es, ev = zip(*el)
            axes[0][col].semilogy(es, ev, 'o-', color='#FF5722', ms=5, lw=1.5)
        axes[0][col].set_title(label, fontsize=10)
        axes[0][col].set_ylabel('Loss' if col == 0 else '')
        axes[0][col].set_xlabel('Step')

        # Row 1: grad norm
        axes[1][col].semilogy(steps, gn, color='#F44336', lw=1.2, alpha=0.8)
        axes[1][col].axhline(1.0, color='gray', linestyle='--', lw=0.8,
                              label='clip=1.0')
        axes[1][col].set_ylabel('Grad Norm' if col == 0 else '')
        axes[1][col].set_xlabel('Step')

        # Row 2: grad/weight ratio
        axes[2][col].semilogy(steps, ra, color='#4CAF50', lw=1.5)
        axes[2][col].axhspan(1e-3, 1e-2, alpha=0.15, color='green')
        axes[2][col].set_ylabel('ρ' if col == 0 else '')
        axes[2][col].set_xlabel('Step')
        axes[2][col].set_ylim(1e-6, 1e0)

    plt.tight_layout()
    plt.savefig(f'{filename}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved {filename}.png")

---

## 1. Loss Plateau

### What the dashboard shows

- Train loss flattens at a value above what the model should be capable of
- Gradient norms are non-zero but small
- Grad/weight ratio is in the range $[10^{-5}, 10^{-4}]$ — well below the healthy band
- LR chart shows either a flat line (no schedule) or has already decayed to near zero

### Why it happens

[[Three distinct causes produce identical-looking loss plateaus.]{.mark} Distinguishing
them requires looking at the ratio chart:

**Cause A — Learning rate too small.** The gradient signal is correct but the
step size is too small to make progress. The ratio $\rho = \|\nabla W\| / \|W\|$
looks healthy, but the effective update $\text{lr} \cdot \rho$ is tiny.

**Cause B — Learning rate decayed too aggressively.** The cosine schedule has
reached its minimum (`min_lr`) while the model still has room to improve.
Visible on the LR chart as a flat line at `min_lr`.

**Cause C — Dead neurons.** A large fraction of FFN neurons output zero for
all inputs. The gradient through dead neurons is exactly zero — those
parameters are frozen. Visible as `act_frac_zero > 0.5` in the activation
monitor.

In [ ]:
# Reproduce a loss plateau — three causes

# Cause A: LR too small
model_a   = GPT(config).to(device)
opt_a     = torch.optim.AdamW(model_a.parameters(), lr=1e-7)   # way too small
res_a     = run_experiment(model_a, opt_a, steps=300, label='lr=1e-7', clip=1.0)

# Cause B: LR decayed to zero
from torch.optim.lr_scheduler import CosineAnnealingLR
model_b   = GPT(config).to(device)
opt_b     = torch.optim.AdamW(model_b.parameters(), lr=3e-4)
sched_b   = CosineAnnealingLR(opt_b, T_max=100, eta_min=0)   # fully decays by step 100

train_b, eval_b, gnorm_b, ratio_b = [], [], [], []
for step in range(300):
    x, y = get_batch()
    _, loss = model_b(x, y)
    opt_b.zero_grad(); loss.backward()
    sched_b.step()
    opt_b.step()
    train_b.append(loss.item())
res_b = (train_b, [], gnorm_b, ratio_b)

# Cause C: Dead neurons (high LR causes weight collapse)
model_c   = GPT(config).to(device)
opt_c     = torch.optim.SGD(model_c.parameters(), lr=10.0)   # absurdly high
res_c     = run_experiment(model_c, opt_c, steps=300, label='dead neurons', clip=0)

plot_pathology(
    {'lr=1e-7': res_a, 'lr decayed to 0': res_b, 'dead neurons': res_c},
    title='Loss Plateau — Three Causes',
    filename='pathology_plateau'
)

### How to diagnose

In [ ]:
def diagnose_plateau(train_losses, grad_norms, ratios, lr_history):
    recent_loss  = np.mean(train_losses[-50:])
    earlier_loss = np.mean(train_losses[-150:-100])
    improvement  = (earlier_loss - recent_loss) / (earlier_loss + 1e-8)

    if improvement < 0.01:   # less than 1% improvement in last 50 steps
        print("PLATEAU DETECTED")

        mean_ratio = np.mean(ratios[-50:])
        current_lr = lr_history[-1] if lr_history else None

        if mean_ratio < 1e-4:
            print("  → Cause: LR too small or fully decayed")
            print(f"    mean ρ={mean_ratio:.2e}  lr={current_lr:.2e}")
            print("    Fix: increase LR, or extend/restart schedule")
        elif current_lr is not None and current_lr < 1e-6:
            print("  → Cause: Schedule decayed to near zero")
            print("    Fix: set min_lr > 0 (typically 10% of max_lr)")
        else:
            print("  → Cause: Possible dead neurons")
            print("    Fix: check act_frac_zero via GradientMonitor")
            print("         consider lower LR, gradient clipping, or LeakyReLU")

### The fix

| Cause | Fix |
|---|---|
| LR too small | Increase by 10× and observe whether ratio enters healthy band |
| Schedule decayed | Set `min_lr = 0.1 * max_lr` (Chinchilla convention) |
| Dead neurons | Reduce LR, add gradient clipping, switch to GELU, or use LeakyReLU |

---

## 2. Loss Spike

### What the dashboard shows

- Loss is decreasing normally, then jumps sharply upward by 0.5–5 nats
- Global gradient norm spikes 1–2 steps *before* the loss spike — the cause
  precedes the effect
- After the spike, loss may recover (partial spike) or stay elevated (bad spike)
- On the ratio chart: one or more layers shows a sudden ratio spike

### Why it happens

A loss spike is caused by a parameter update that moves a weight too far from
its current value — the model jumps to a region of higher loss. The gradient
norm spike is the precursor: a single batch with unusually high loss generates
a large gradient, the optimizer takes a large step, and the model lands
somewhere worse.

Two triggers:

**Trigger A — Adversarial batch.** A batch that happens to have unusually high
perplexity (very rare token combinations, very long dependencies). Produces a
legitimate but large gradient. Gradient clipping prevents the step from being
catastrophic.

**Trigger B — LR too high relative to current loss landscape.** As loss
decreases and the model moves into a sharper region of the loss surface,
the same LR that was fine early in training becomes too large. The model
oscillates around a minimum rather than converging into it. Requires LR
decay to fix.

In [ ]:
# Reproduce a loss spike with an adversarial batch injection

model = GPT(config).to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_losses, grad_norms = [], []

for step in range(300):
    x, y = get_batch()

    # At step 150, inject a "pathological" batch:
    # repeat the same rare token to create an unusually hard prediction task
    if step == 150:
        rare_token = config.vocab_size - 1   # last token in vocab
        x = torch.full((8, block), rare_token, dtype=torch.long, device=device)
        y = torch.randint(0, config.vocab_size, (8, block), device=device)
        print(f"  Injecting adversarial batch at step {step}")

    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()

    gnorm = sum(p.grad.norm().item()**2 for p in model.parameters()
                if p.grad is not None) ** 0.5
    grad_norms.append(gnorm)

    # No clipping — let the spike happen
    opt.step()
    train_losses.append(loss.item())

# Now repeat WITH clipping
model_clipped = GPT(config).to(device)
opt_clipped   = torch.optim.AdamW(model_clipped.parameters(), lr=3e-4)
train_clipped, _, gnorm_clipped, _ = run_experiment(
    model_clipped, opt_clipped, steps=300, clip=1.0
)
# Manually inject at step 150 in the clipped version is left as Exercise 1

### Reading the timing

[The key diagnostic: plot `grad_norm[t]` and `train_loss[t+1]` on the same
axis.]{.underline} If the gradient norm spike at step $t$ correlates with the loss spike
at step $t+1$, you have confirmed the causal chain:

In [ ]:
def find_spikes(grad_norms, threshold_multiplier=5.0):
    """Find steps where grad norm exceeds threshold_multiplier × rolling mean."""
    norms  = np.array(grad_norms)
    window = 50
    spikes = []
    for i in range(window, len(norms)):
        mean = norms[max(0, i-window):i].mean()
        if norms[i] > threshold_multiplier * mean:
            spikes.append(i)
    return spikes

spikes = find_spikes(grad_norms)
print(f"Gradient norm spikes at steps: {spikes}")
# Check if loss spiked 1-2 steps later:
for s in spikes:
    if s + 2 < len(train_losses):
        print(f"  step {s}: gnorm={grad_norms[s]:.2f}  "
              f"loss[+1]={train_losses[s+1]:.4f}  "
              f"loss[+2]={train_losses[s+2]:.4f}")

### The fix

```python
# Standard: clip_grad_norm_ with max_norm=1.0
# This is the single most effective spike prevention measure.
gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# If spikes persist after clipping, the LR is too high.
# Reduce LR by 3-5x and observe whether the spike frequency drops.

# If a spike is so large it corrupts the model (loss stays elevated
# after the spike), reload the last checkpoint:
torch.save(model.state_dict(), 'checkpoint.pt')   # save regularly
# ...if spike corrupts the run:
model.load_state_dict(torch.load('checkpoint.pt'))
```

---

## 3. Loss Divergence

### What the dashboard shows

- Loss grows monotonically — never recovers
- Gradient norms grow monotonically — each step is worse than the last
- Ratio chart shows all layers in the red zone ($\rho > 10^{-1}$)
- GPU memory may spike if intermediate tensors grow large

### Why it happens

[[Divergence is a positive feedback loop:]{.mark}

$$\text{large loss} \rightarrow \text{large gradient} \rightarrow \text{large update} \rightarrow \text{larger loss} \rightarrow \ldots$$

Once this loop starts, it accelerates. Three root causes:

**Cause A — LR too large.** The most common cause. The optimizer overshoots
every minimum. Weights grow without bound.

**Cause B — Missing gradient clipping.** [A single adversarial batch produces
a gradient so large that one step corrupts the model permanently.]{.mark}

**Cause C — Bad initialization.** Weights initialized too large cause
activation saturation immediately. Gradients through saturated regions
are near zero — the model cannot recover from its initial state and
the loss is stuck at a high value that grows as the optimizer makes
random-walk steps.

In [ ]:
# Reproduce divergence — cause A: LR too large

model_div = GPT(config).to(device)
opt_div   = torch.optim.AdamW(model_div.parameters(), lr=1.0)  # 3000× too large
res_div   = run_experiment(model_div, opt_div, steps=100,
                           label='lr=1.0 (diverging)', clip=0)

# Reproduce divergence — cause C: bad init

model_badinit = GPT(config).to(device)
# Reinitialize all weights with very large std
for m in model_badinit.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 2.0)   # 100× too large
opt_badinit = torch.optim.AdamW(model_badinit.parameters(), lr=3e-4)
res_badinit = run_experiment(model_badinit, opt_badinit, steps=100,
                              label='init std=2.0', clip=0)

plot_pathology(
    {'lr=1.0 (no clip)': res_div, 'bad init (std=2.0)': res_badinit},
    title='Loss Divergence',
    filename='pathology_divergence'
)

### The fix

Divergence is usually caught in the first 50 steps. If loss has not started
decreasing by step 50 and gradient norms are growing:

1. [Kill the run immediately — do not let it continue]{.underline}
2. Reduce LR by 10× and restart
3. If divergence persists: check initialization (initial loss should be
   $\approx \log(\text{vocab\_size}) \approx 10.8$ for our tokenizer)
4. Add gradient clipping (`max_norm=1.0`) if not already present

In [ ]:
# Early divergence detector — add to the training loop
def check_divergence(step, loss, initial_loss, threshold=3.0):
    """
    Returns True if loss has diverged.
    Trigger: loss is more than `threshold`× the initial loss after warmup.
    """
    if step < 20:
        return False   # ignore during warmup
    if loss > threshold * initial_loss:
        print(f"  DIVERGENCE at step {step}: "
              f"loss={loss:.4f} > {threshold}×initial={initial_loss:.4f}")
        return True
    return False

---

## 4. Vanishing Gradients

### What the dashboard shows

- Train loss decreases but slowly — much slower than expected
- The per-layer ratio chart shows a gradient: early layers near $10^{-6}$,
  final layers near $10^{-2}$
- Weight norms for early layers are flat over time — they are not updating
- The gradient norm vs layer depth plot (from Tutorial 4) shows an
  exponential decay toward early layers

### Why it happens

Without residual connections, the gradient decomposes into a product of
Jacobians across layers:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \frac{\partial x_{i+1}}{\partial x_i}$$

Each Jacobian has spectral norm $\leq 1$ for typical activations. The product
shrinks exponentially with the number of layers between $l$ and $L$.

For the Transformer with residual connections, this should not happen —
the $\left(I + \frac{\partial F}{\partial x}\right)$ term guarantees the
gradient highway (Tutorial 2, Section 6). If you are seeing vanishing
gradients in a Transformer, the usual culprits are:

- Pre-norm disabled (post-norm allows gradient attenuation through LN)
- Residual scaling missing (Tutorial 2, Section 9)
- Attention softmax saturating (attention weights collapse to one-hot)

In [ ]:
# Reproduce vanishing gradients: deep net without residuals
# (using a plain MLP, not the GPT, to make the effect stark)

class DeepMLP(nn.Module):
    def __init__(self, depth=12, d=256, use_residual=False):
        super().__init__()
        self.use_residual = use_residual
        self.layers = nn.ModuleList([
            nn.Linear(d, d) for _ in range(depth)
        ])
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        for layer in self.layers:
            h = torch.tanh(layer(x))
            x = x + h if self.use_residual else h
        return self.head(x)

def measure_layer_gradients(model, x):
    """Returns per-layer gradient norm after one backward pass."""
    y    = model(x).mean()
    y.backward()
    norms = []
    for layer in model.layers:
        if layer.weight.grad is not None:
            norms.append(layer.weight.grad.norm().item())
        else:
            norms.append(0.0)
    return norms

x = torch.randn(32, 256, requires_grad=False)

# Without residuals
mlp_no_res  = DeepMLP(depth=12, use_residual=False)
norms_no_res = measure_layer_gradients(mlp_no_res, x)

# With residuals
mlp_res     = DeepMLP(depth=12, use_residual=True)
norms_res   = measure_layer_gradients(mlp_res, x)

# Plot: per-layer gradient norm
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(norms_no_res,  'o-', color='#F44336', lw=2, label='no residuals')
ax.semilogy(norms_res,     's-', color='#4CAF50', lw=2, label='with residuals')
ax.set_xlabel('Layer index (0 = earliest)')
ax.set_ylabel('Gradient norm (log)')
ax.set_title('Vanishing Gradients: Residual vs No-Residual (12-layer MLP)')
ax.axhline(1e-3, color='gray', linestyle='--', lw=0.8)
ax.legend()
plt.tight_layout()
plt.savefig('vanishing_gradients.png', dpi=150)
plt.show()

### The fix

For the Transformer specifically:

In [ ]:
# 1. Verify residual connections are present in every block
# 2. Verify pre-norm (not post-norm) is being used
# 3. Verify residual scaling is applied at init
# 4. Check attention saturation:

def check_attention_saturation(model, x):
    """
    Run a forward pass and collect attention weight entropy per head.
    Low entropy = saturated (one token dominates).
    """
    entropies = []
    hooks     = []

    def attn_hook(module, inp, out):
        # out is (attn_output, attn_weights) from our MultiHeadAttention
        if isinstance(out, tuple) and len(out) == 2:
            weights = out[1]   # (B, H, T, T)
            # Entropy of attention distribution per head
            eps = 1e-8
            ent = -(weights * (weights + eps).log()).sum(-1).mean().item()
            entropies.append(ent)

    for block in model.blocks:
        hooks.append(block.attn.register_forward_hook(attn_hook))

    with torch.no_grad():
        model(x)

    for h in hooks:
        h.remove()

    print(f"Attention entropy per block (higher = more distributed):")
    for i, e in enumerate(entropies):
        bar = '█' * int(e * 5)
        print(f"  block {i:2d}: {e:.3f}  {bar}")

---

## 5. Exploding Gradients

### What the dashboard shows

- Gradient norm grows rapidly — often 10×–100× in a few steps
- Loss may initially decrease (the large gradient is in the right direction)
  then suddenly spike as the large update overshoots
- Ratio chart shows all layers deep red ($\rho > 1.0$)
- Without clipping: model weights grow to `inf`, loss becomes `nan`

### Why it happens

Exploding gradients are the mirror image of vanishing: instead of the Jacobian
product shrinking, it grows. This happens when weight matrices have spectral
norm $> 1$, which amplifies the gradient at each layer.

The mathematical condition for stable gradient flow is that the spectral norm
of each layer's Jacobian is $\approx 1$ — exactly what good initialization
achieves. As training proceeds and weights move away from init, this balance
can be broken if the learning rate is too large or the loss surface is very
nonlinear.

In [ ]:
# Reproduce exploding gradients: large init + no clipping

model_exp = GPT(config).to(device)
for m in model_exp.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.5)   # 25× too large

opt_exp = torch.optim.AdamW(model_exp.parameters(), lr=3e-4)
res_exp_noclip = run_experiment(
    model_exp, opt_exp, steps=100,
    label='large init, no clip', clip=0
)

# Same model, with clipping — shows clipping as the fix
model_exp2 = GPT(config).to(device)
for m in model_exp2.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.5)
opt_exp2 = torch.optim.AdamW(model_exp2.parameters(), lr=3e-4)
res_exp_clipped = run_experiment(
    model_exp2, opt_exp2, steps=100,
    label='large init, clip=1.0', clip=1.0
)

plot_pathology(
    {'no clipping': res_exp_noclip, 'with clipping': res_exp_clipped},
    title='Exploding Gradients — Clipping as the Fix',
    filename='pathology_exploding'
)

### Choosing the right clip value

`max_norm=1.0` is the standard starting point. Here is how to tune it:

In [ ]:
def calibrate_clip_threshold(model, optimizer, dataloader, n_steps=200):
    """
    Run training for n_steps without clipping.
    Return the 95th percentile of pre-clip gradient norms.
    Set clip = that value.
    """
    norms = []
    for step, (x, y) in enumerate(dataloader):
        if step >= n_steps:
            break
        _, loss = model(x.to(device), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        # Compute norm without clipping
        total = sum(p.grad.norm().item()**2 for p in model.parameters()
                    if p.grad is not None) ** 0.5
        norms.append(total)
        optimizer.step()

    p95 = float(np.percentile(norms, 95))
    print(f"95th percentile grad norm over {n_steps} steps: {p95:.3f}")
    print(f"Recommended clip threshold: {p95:.2f}")
    return p95

Set `max_norm` to the 95th percentile of the unclipped gradient norm
distribution. This means clipping fires on roughly 5% of steps — the
outliers — and has no effect on the other 95%.

---

## 6. NaN Propagation

### What the dashboard shows

- Loss becomes exactly `nan` — not a large number, `nan`
- All subsequent losses are `nan` — once you have a NaN in the weights,
  every forward pass produces NaN
- Gradient norms are `nan` before the loss becomes `nan`

### Why it happens

NaN propagates through any arithmetic operation that receives it as input:
`nan + x = nan`, `nan * x = nan`, `nan > x = False`. [Once any weight
contains NaN, the entire model is dead.]{.mark}

Three common sources:

**Source A — `log(0)`.** Cross-entropy loss computes $-\log p$ where $p$ is
a softmax probability. If the logit for the true token is $-\infty$ (which
can happen with exploding gradients pushing logits to very large negative
values), the softmax probability is 0 and $\log(0) = -\infty$. The gradient
of $-\infty$ is `nan`.

**Source B — `0/0` in attention.** If all attention logits for a query
position are $-\infty$ (e.g., a fully masked row in the causal mask for
a sequence of length 1), `softmax([-inf, -inf, ...])` produces `nan`
because $e^{-\infty} / \sum e^{-\infty} = 0/0$.

**Source C — Overflow in `exp`.** `exp(89)` overflows to `inf` in float32.
`inf - inf = nan`. Can occur in the softmax computation when logits are
very large.

In [ ]:
# Reproduce NaN from log(0)

# A single logit pushed to -inf causes the NaN
logits = torch.tensor([[0.0, -1e38, 0.0]])   # middle logit = -inf effectively
target = torch.tensor([1])                    # predict the -1e38 class

loss = F.cross_entropy(logits, target)
print(f"loss: {loss}")   # tensor(nan) or very large number

# The safe fix: check for NaN in the loss before backward
if torch.isnan(loss) or torch.isinf(loss):
    print("Skipping step — NaN/Inf loss detected")
    optimizer.zero_grad()
    continue

### NaN detection and recovery

In [ ]:
def safe_backward(loss, optimizer, model, step):
    """
    Backward pass with NaN detection.
    Returns True if the step was taken, False if it was skipped.
    """
    if torch.isnan(loss) or torch.isinf(loss):
        print(f"  Step {step}: NaN/Inf loss={loss.item()} — skipping")
        optimizer.zero_grad()
        return False

    optimizer.zero_grad()
    loss.backward()

    # Check for NaN in gradients
    nan_params = []
    for name, p in model.named_parameters():
        if p.grad is not None and (torch.isnan(p.grad).any() or
                                    torch.isinf(p.grad).any()):
            nan_params.append(name)

    if nan_params:
        print(f"  Step {step}: NaN gradient in {nan_params[:3]}... — skipping")
        optimizer.zero_grad()
        return False

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return True

### The fix for attention NaN

Add a small epsilon to the softmax denominator, or use PyTorch's built-in
`F.scaled_dot_product_attention` which handles numerical stability internally:

In [ ]:
# In your attention implementation — guard against all-masked rows
def safe_softmax(scores, mask=None):
    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)   # use -1e9, not -inf
        # -inf causes nan when all positions are masked;
        # -1e9 produces a very peaked but valid distribution

    # For numerical stability: subtract max before exp
    scores = scores - scores.max(dim=-1, keepdim=True).values
    return F.softmax(scores, dim=-1)

---

## 7. Dead Neurons

### What the dashboard shows

- `act_frac_zero` for one or more FFN layers climbs above 0.5 and stays there
- Loss plateaus — the layer has lost half or more of its capacity
- Grad norm for that layer's weights is near zero (dead units do not
  contribute to the gradient)
- The plateau is permanent — unlike a loss spike, dead neurons do not recover

### Why they happen

A ReLU neuron is dead when its pre-activation is negative for every input
in the dataset. The gradient through `max(0, x)` at `x < 0` is exactly 0.[^relu_zero]

[^relu_zero]: The derivative of $\max(0, x)$ is 1 for $x > 0$ and 0 for $x < 0$. Once a neuron's pre-activation is negative for every input in the dataset, the chain rule multiplies every downstream gradient by 0 — the parameter receives no update and is permanently frozen.
Once dead, the weights never receive a gradient and never recover.

Triggers: a single very large gradient step (LR spike, missing clip) that
pushes the weights into the permanently-negative regime. A large negative
bias is the most common mechanism — if the bias is pushed to $-10$, the
pre-activation is $W^\top x + b \approx W^\top x - 10$ which is negative
for almost all inputs.

GELU mitigates but does not eliminate this: GELU has a small gradient for
$x \ll 0$ (approximately `x * N(x; 0, 1)` which is very small but nonzero).
[In practice, GELU neurons are much harder to kill permanently.]{.mark}

In [ ]:
def count_dead_neurons(model, dataloader, n_batches=10, threshold=0.99):
    """
    For each Linear layer followed by an activation, count the fraction
    of neurons that output near-zero for all inputs in n_batches.

    threshold=0.99 means a neuron is 'dead' if it outputs near-zero
    on 99% of tokens across all batches.
    """
    activation_stats = {}  # name → running fraction zero

    hooks = []
    def make_hook(name):
        def hook(module, inp, out):
            # out: (B, T, d) or (B, d)
            flat = out.detach().reshape(-1, out.shape[-1])
            # Fraction of positions where this neuron is near-zero
            frac = (flat.abs() < 0.01).float().mean(0)   # (d,)
            if name not in activation_stats:
                activation_stats[name] = frac
            else:
                activation_stats[name] += frac
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):
            if i >= n_batches:
                break
            model(x.to(device))

    for h in hooks:
        h.remove()

    print("\nDead neuron analysis:")
    for name, frac_sum in activation_stats.items():
        frac_mean = frac_sum / n_batches
        dead_frac  = (frac_mean > threshold).float().mean().item()
        if dead_frac > 0.1:
            print(f"  ⚠ {name:50s}  {dead_frac:.1%} dead neurons")
        else:
            print(f"  ✓ {name:50s}  {dead_frac:.1%} dead neurons")

### The fix

[Dead neurons cannot be revived by gradient descent. Prevention is the only
cure:]{.underline}

```python
# Prevention 1: Use GELU instead of ReLU
# GELU has nonzero gradient everywhere — neurons cannot be permanently killed

# Prevention 2: Gradient clipping stops the large-step that kills neurons
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

# Prevention 3: Weight regularization keeps weights from drifting too far
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

# Prevention 4: If you must use ReLU and see dead neurons, use LeakyReLU
# which has a small negative slope (default 0.01) for x < 0
nn.LeakyReLU(negative_slope=0.01)
```

---

## 8. Overfitting

### What the dashboard shows

- Train loss continues to decrease
- Eval loss stops decreasing and starts increasing — the gap widens
- Gradient norms are healthy — the model is learning, just the wrong thing
- The train/eval loss gap is the only signal — everything else looks fine

### Why it happens

The model has memorized the training set and is no longer learning
generalisable patterns. For language models this is more subtle than
for classification: a model that memorizes training text will have low
train loss but will assign high perplexity to held-out text because
it has overfit to specific sequences.

For our nano model on TinyShakespeare, overfitting happens quickly because
the dataset is small (~1M tokens). After a few epochs through the data,
the model has essentially memorized the training split.

In [ ]:
# Compute and plot the train/eval gap over time

model    = GPT(config).to(device)
opt      = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

train_losses, eval_losses = [], []
eval_steps = []

for step in range(2000):
    x, y   = get_batch('train')
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    train_losses.append(loss.item())

    if step % 100 == 0:
        el = evaluate(model)
        eval_losses.append(el)
        eval_steps.append(step)
        gap = el - np.mean(train_losses[-50:])
        print(f"step {step:4d}  train={train_losses[-1]:.4f}  "
              f"eval={el:.4f}  gap={gap:+.4f}")

# The gap is your stopping criterion
# Optimal stopping = step where eval loss is minimum
best_step = eval_steps[np.argmin(eval_losses)]
print(f"\nBest eval loss at step {best_step}")

### The fix

Overfitting is managed, not eliminated. The main levers:

In [ ]:
# 1. More data — the most effective fix, always
# If you're using TinyShakespeare (~1M tokens), switch to a larger corpus

# 2. Weight decay — L2 regularization via AdamW
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=3e-4,
                               weight_decay=0.1)   # standard for LMs

# 3. Early stopping — stop when eval loss starts rising
best_eval = float('inf')
patience  = 5   # number of eval checks without improvement before stopping
no_improve = 0

if eval_loss < best_eval:
    best_eval   = eval_loss
    no_improve  = 0
    torch.save(model.state_dict(), 'best_model.pt')
else:
    no_improve += 1
    if no_improve >= patience:
        print(f"Early stopping at step {step}")
        break

# 4. Chinchilla-optimal training: do not over-train on a small dataset
# Rule of thumb: train for D ≈ 20×N tokens where N = model parameters
# For our 10.7M nano model: D ≈ 214M tokens
# TinyShakespeare has ~1M tokens — stop after 1 epoch!

---

## 9. The `HealthReport`

Everything above, automated. Run at the end of each epoch (or every 500 steps)
and get a prioritized list of what to fix:

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class Severity(Enum):
    OK      = "✓"
    WARNING = "⚠"
    CRITICAL = "✗"

@dataclass
class Finding:
    severity:  Severity
    check:     str
    detail:    str
    fix:       str

class HealthReport:
    """
    Runs a battery of diagnostics on the recorded training history
    and produces a prioritized list of findings.

    Usage:
        report = HealthReport()

        # Inside training loop:
        report.record(
            step=step,
            train_loss=loss.item(),
            eval_loss=eval_loss,       # None if not computed this step
            grad_norm=gnorm,
            grad_ratio=mean_ratio,
            lr=current_lr,
            act_frac_zero=frac_zero,   # from GradientMonitor, optional
        )

        # Every epoch:
        report.print_report()
    """

    def __init__(self, window: int = 100):
        self.window = window
        self._train_loss:  list[float] = []
        self._eval_loss:   list[tuple] = []   # (step, loss)
        self._grad_norm:   list[float] = []
        self._grad_ratio:  list[float] = []
        self._lr:          list[float] = []
        self._act_zero:    list[float] = []
        self._step:        int         = 0
        self._initial_loss: float | None = None

    def record(
        self,
        step: int,
        train_loss: float,
        grad_norm: float,
        grad_ratio: float,
        lr: float,
        eval_loss: float = None,
        act_frac_zero: float = None,
    ):
        self._step = step
        self._train_loss.append(train_loss)
        self._grad_norm.append(grad_norm)
        self._grad_ratio.append(grad_ratio)
        self._lr.append(lr)
        if eval_loss is not None:
            self._eval_loss.append((step, eval_loss))
        if act_frac_zero is not None:
            self._act_zero.append(act_frac_zero)
        if self._initial_loss is None and not np.isnan(train_loss):
            self._initial_loss = train_loss

    def _recent(self, lst, n=None):
        n = n or self.window
        return lst[-n:] if lst else []

    def run_checks(self) -> list[Finding]:
        findings = []
        tl   = self._recent(self._train_loss)
        gn   = self._recent(self._grad_norm)
        gr   = self._recent(self._grad_ratio)
        lr   = self._recent(self._lr)
        az   = self._recent(self._act_zero)

        if len(tl) < 10:
            return [Finding(Severity.OK, "Insufficient data",
                            "Need at least 10 steps", "Keep training")]

        # --- NaN check ---
        if any(np.isnan(v) for v in tl):
            findings.append(Finding(
                Severity.CRITICAL, "NaN loss",
                "Loss is NaN — model is dead",
                "Reload checkpoint, add NaN guard, reduce LR"
            ))
            return findings   # no point checking further

        # --- Divergence check ---
        if np.isnan(np.mean(tl)):
            pass
        elif len(tl) >= 20 and (self._initial_loss is not None
                                 and np.mean(tl[-20:]) > 2.0 * self._initial_loss):
            findings.append(Finding(
                Severity.CRITICAL, "Divergence",
                f"Loss {np.mean(tl[-20:]):.4f} > 2× initial {self._initial_loss:.4f}",
                "Kill run, reduce LR by 10×, restart"
            ))

        # --- Plateau check ---
        if len(tl) >= 50:
            recent   = np.mean(tl[-25:])
            earlier  = np.mean(tl[-75:-25])
            improvement = (earlier - recent) / (earlier + 1e-8)
            if improvement < 0.005:
                mean_ratio = np.mean(gr) if gr else 0
                if mean_ratio < 1e-4:
                    fix = "LR too small or decayed — increase LR or extend schedule"
                elif lr and lr[-1] < 1e-6:
                    fix = "Schedule fully decayed — set min_lr > 0 (10% of max_lr)"
                elif az and np.mean(az) > 0.4:
                    fix = "Dead neurons — use GELU, add grad clipping, reduce LR"
                else:
                    fix = "Unknown cause — check activation histograms"
                findings.append(Finding(
                    Severity.WARNING, "Loss plateau",
                    f"<0.5% improvement over last 50 steps (ratio={mean_ratio:.2e})",
                    fix
                ))

        # --- Exploding gradients ---
        if gn:
            mean_gn = np.mean(gn)
            max_gn  = np.max(gn)
            if max_gn > 10 * mean_gn:
                findings.append(Finding(
                    Severity.WARNING, "Gradient spikes",
                    f"Max grad norm {max_gn:.2f} = {max_gn/mean_gn:.1f}× mean",
                    "Add/reduce clip threshold; check for outlier batches"
                ))
            if mean_gn > 10.0:
                findings.append(Finding(
                    Severity.CRITICAL, "Exploding gradients",
                    f"Mean grad norm {mean_gn:.2f} >> 1.0",
                    "Reduce LR by 5×, add gradient clipping max_norm=1.0"
                ))

        # --- Vanishing gradients ---
        if gr:
            mean_ratio = np.mean(gr)
            if mean_ratio < 1e-5:
                findings.append(Finding(
                    Severity.WARNING, "Vanishing gradients",
                    f"Mean ρ={mean_ratio:.2e} << healthy [1e-3, 1e-2]",
                    "Check residual connections, LN placement, init"
                ))

        # --- Unhealthy ratio ---
        if gr:
            mean_ratio = np.mean(gr)
            if 1e-5 <= mean_ratio < 1e-3:
                findings.append(Finding(
                    Severity.WARNING, "Ratio below healthy band",
                    f"Mean ρ={mean_ratio:.2e} (healthy: 1e-3 to 1e-2)",
                    "Increase LR slightly or check LR schedule"
                ))
            elif mean_ratio > 1e-1:
                findings.append(Finding(
                    Severity.WARNING, "Ratio above healthy band",
                    f"Mean ρ={mean_ratio:.2e} (healthy: 1e-3 to 1e-2)",
                    "Reduce LR or add gradient clipping"
                ))

        # --- Overfitting ---
        if len(self._eval_loss) >= 3:
            eval_vals = [v for _, v in self._eval_loss]
            train_recent = np.mean(tl[-25:])
            eval_recent  = eval_vals[-1]
            gap = eval_recent - train_recent
            # Check if eval is getting worse
            if len(eval_vals) >= 3 and eval_vals[-1] > eval_vals[-2] > eval_vals[-3]:
                findings.append(Finding(
                    Severity.WARNING, "Overfitting",
                    f"Eval loss rising for 3 consecutive checks. Gap={gap:.4f}",
                    "More data, weight decay, or early stopping"
                ))

        # --- Dead neurons ---
        if az and np.mean(az) > 0.3:
            findings.append(Finding(
                Severity.WARNING, "Dead neurons",
                f"Mean activation zero fraction={np.mean(az):.1%}",
                "Switch to GELU, add clipping, reduce LR"
            ))

        # --- All clear ---
        if not findings:
            findings.append(Finding(
                Severity.OK, "All checks passed",
                f"step={self._step}  loss={np.mean(tl[-10:]):.4f}  "
                f"ρ={np.mean(gr):.2e}",
                "Keep training"
            ))

        return findings

    def print_report(self):
        findings = self.run_checks()
        # Sort: CRITICAL first, then WARNING, then OK
        order = {Severity.CRITICAL: 0, Severity.WARNING: 1, Severity.OK: 2}
        findings.sort(key=lambda f: order[f.severity])

        print(f"\n{'═'*60}")
        print(f"  Health Report — step {self._step}")
        print(f"{'═'*60}")
        for f in findings:
            print(f"\n  {f.severity.value}  {f.check}")
            print(f"     {f.detail}")
            print(f"     Fix: {f.fix}")
        print(f"\n{'═'*60}\n")

Usage in the training loop:

In [ ]:
report = HealthReport()

for step in range(max_steps):
    x, y   = get_batch()
    _, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()
    optimizer.step()

    # Per-step stats
    ratios = [p.grad.norm().item() / (p.norm().item() + 1e-8)
              for p in model.parameters() if p.grad is not None]
    mean_ratio = float(np.mean(ratios)) if ratios else 0.0

    report.record(
        step=step,
        train_loss=loss.item(),
        grad_norm=gnorm,
        grad_ratio=mean_ratio,
        lr=optimizer.param_groups[0]['lr'],
    )

    # Run report every 500 steps
    if step % 500 == 0 and step > 0:
        report.print_report()

Example output:

```
════════════════════════════════════════════════════════════
  Health Report — step 500
════════════════════════════════════════════════════════════

  ⚠  Loss plateau
     <0.5% improvement over last 50 steps (ratio=8.21e-05)
     Fix: LR too small or decayed — increase LR or extend schedule

  ✓  All checks passed
     step=500  loss=3.1204  ρ=8.21e-05
     Fix: Keep training

════════════════════════════════════════════════════════════
```

---

## Summary

| Pathology | Primary signal | Root cause | Fix |
|---|---|---|---|
| Loss plateau | Flat loss, $\rho < 10^{-4}$ | LR too small, decayed, or dead neurons | Increase LR, extend schedule, use GELU |
| Loss spike | Grad norm spike → loss spike | Adversarial batch, LR too high | Gradient clipping, reduce LR |
| Loss divergence | Monotonically growing loss | LR way too high, bad init | Reduce LR 10×, fix init, add clipping |
| Vanishing gradients | Per-layer ratio decays with depth | No residuals, post-norm, missing $1/\sqrt{2L}$ scaling | Add residuals, switch to pre-norm |
| Exploding gradients | $\rho > 10^{-1}$, growing grad norm | Bad init, LR too high | Grad clipping, fix init, reduce LR |
| NaN propagation | Loss = NaN, all weights dead | `log(0)`, softmax overflow, exploding grads | NaN guard, use `-1e9` mask, clip |
| Dead neurons | `act_frac_zero > 0.5`, plateau | Large LR step pushes biases negative | GELU, clipping, weight decay |
| Overfitting | Eval loss rising, train loss falling | Insufficient data, too many steps | More data, weight decay, early stopping |

---

## Exercises

**1.** Reproduce the loss spike experiment with gradient clipping enabled
(`clip=1.0`). Inject the adversarial batch at step 150. Plot the gradient
norm before and after the injection with and without clipping on the same
axis. Confirm that clipping prevents the spike in the loss curve.

**2.** Implement a `checkpoint_on_spike` function that saves `model.state_dict()`
whenever `find_spikes()` detects a gradient norm spike. Then reload the
checkpoint and continue training. Measure how many steps it takes to
recover the pre-spike loss vs training through the spike.

**3.** Add a check to `HealthReport` for **learning rate warmup not completed**:
if the LR history shows the LR is still in the warmup phase (still
increasing linearly) after more than 10% of `max_steps`, flag it as a
warning. Add a `max_steps` argument to `HealthReport.__init__`.

**4.** The `diagnose_plateau` function distinguishes three causes by looking
at `mean_ratio` and `current_lr`. Add a fourth branch: if `mean_ratio`
is healthy but loss is still flat, suggest that the model may have
reached the Chinchilla-optimal point for the dataset size and training
beyond this is counterproductive.

**5.** Extend `count_dead_neurons` to also detect **saturated neurons** —
units whose pre-activation is always in the flat (high-confidence) region
of the activation function. For GELU, this means `abs(pre_act) > 3.0`
for more than 90% of inputs. Add a hook to capture pre-activation values
(before the nonlinearity) and compute this fraction.

**6.** The `HealthReport` checks the mean gradient-to-weight ratio across all
layers. Extend it to check the *per-layer* ratio and flag any individual
layer whose ratio is more than 10× below the mean — this identifies a
specific layer that is not learning, rather than a global problem.